# 01 Data Preprocessing

Preprocess the Jigsaw toxic comment data, create binary targets, run exploratory analysis, prepare the labeled held-out test set, and export the processed CSVs used by the model-training notebook.

In [1]:
import os
from pathlib import Path
import html
import re
import subprocess
import sys

try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    colab_drive = None
    IN_COLAB = False

# Update this path if you store the project in a different Google Drive folder.
COLAB_PROJECT_DIR = Path("/content/drive/MyDrive/BT5151_toxic_comment_agent")
AUTO_MOUNT_DRIVE = True

if IN_COLAB and AUTO_MOUNT_DRIVE:
    colab_drive.mount("/content/drive", force_remount=False)

if IN_COLAB:
    if not COLAB_PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"Colab project directory not found: {COLAB_PROJECT_DIR}. "
            "Upload the full repository folder to Google Drive and update COLAB_PROJECT_DIR if needed."
        )
    os.chdir(COLAB_PROJECT_DIR)

requirements_path = Path("experiments/requirements-experiments.txt")
if IN_COLAB and requirements_path.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
PROCESSED_COLUMNS = ["id", "comment_text_clean", "toxic_label"]

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 120

def resolve_root_dir(start: Path) -> Path:
    start = start.resolve()
    if start.name == "experiments" and (start.parent / "raw_data").exists():
        return start.parent
    if (start / "raw_data").exists():
        return start
    if (start / "experiments" / "raw_data").exists():
        return start
    if (start.parent / "raw_data").exists():
        return start.parent
    raise FileNotFoundError("Could not locate repo root from the current working directory.")

ROOT_DIR = resolve_root_dir(Path.cwd())
EXPERIMENTS_DIR = ROOT_DIR / "experiments"
RAW_DATA_DIR = ROOT_DIR / "raw_data"
PROCESSED_DIR = EXPERIMENTS_DIR / "processed_data"
FIGURES_DIR = EXPERIMENTS_DIR / "figures"

for directory in (PROCESSED_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Running in Colab: {IN_COLAB}")
print(f"Working directory: {Path.cwd().resolve()}")
print(f"ROOT_DIR: {ROOT_DIR}")
print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab: True
Working directory: /content/drive/MyDrive/BT5151_toxic_comment_agent
ROOT_DIR: /content/drive/MyDrive/BT5151_toxic_comment_agent
RAW_DATA_DIR: /content/drive/MyDrive/BT5151_toxic_comment_agent/raw_data
PROCESSED_DIR: /content/drive/MyDrive/BT5151_toxic_comment_agent/experiments/processed_data
FIGURES_DIR: /content/drive/MyDrive/BT5151_toxic_comment_agent/experiments/figures


## Raw Data Load

In [2]:
train_raw = pd.read_csv(RAW_DATA_DIR / "train.csv")
test_raw = pd.read_csv(RAW_DATA_DIR / "test.csv")
test_labels_raw = pd.read_csv(RAW_DATA_DIR / "test_labels.csv")

for name, df in [("train_raw", train_raw), ("test_raw", test_raw), ("test_labels_raw", test_labels_raw)]:
    print(f"{name}: shape={df.shape}")
    print(df.dtypes)
    print(f"null counts (top 10):\n{df.isna().sum().sort_values(ascending=False).head(10)}")
    print(f"duplicate rows: {df.duplicated().sum()}")
    print()

print("train_raw preview:")
print(train_raw.head(3).to_string(index=False))

train_raw: shape=(159571, 8)
id               object
comment_text     object
toxic             int64
severe_toxic      int64
obscene           int64
threat            int64
insult            int64
identity_hate     int64
dtype: object
null counts (top 10):
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64
duplicate rows: 0

test_raw: shape=(153164, 2)
id              object
comment_text    object
dtype: object
null counts (top 10):
id              0
comment_text    0
dtype: int64
duplicate rows: 0

test_labels_raw: shape=(153164, 7)
id               object
toxic             int64
severe_toxic      int64
obscene           int64
threat            int64
insult            int64
identity_hate     int64
dtype: object
null counts (top 10):
id               0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: in

## Label Creation

In [3]:
def add_binary_target(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["toxic_label"] = out[LABEL_COLS].gt(0).any(axis=1).astype(int)
    return out

def label_cooccurrence_matrix(df: pd.DataFrame) -> pd.DataFrame:
    label_bin = df[LABEL_COLS].gt(0).astype(int)
    matrix = label_bin.T.dot(label_bin)
    return matrix.astype(int)

train_labeled = add_binary_target(train_raw)
train_label_counts = train_labeled["toxic_label"].value_counts().sort_index()
train_label_rates = train_labeled["toxic_label"].value_counts(normalize=True).sort_index()

print("Binary target distribution in the full training set:")
print(pd.DataFrame({"count": train_label_counts, "rate": train_label_rates}).rename(index={0: "clean", 1: "toxic"}))
print()
print("Six-label positives in the original training data:")
print(train_raw[LABEL_COLS].sum().sort_values(ascending=False))

co_occurrence = label_cooccurrence_matrix(train_raw)
print("Label co-occurrence matrix:")
print(co_occurrence)

Binary target distribution in the full training set:
              count      rate
toxic_label                  
clean        143346  0.898321
toxic         16225  0.101679

Six-label positives in the original training data:
toxic            15294
obscene           8449
insult            7877
severe_toxic      1595
identity_hate     1405
threat             478
dtype: int64
Label co-occurrence matrix:
               toxic  severe_toxic  obscene  threat  insult  identity_hate
toxic          15294          1595     7926     449    7344           1302
severe_toxic    1595          1595     1517     112    1371            313
obscene         7926          1517     8449     301    6155           1032
threat           449           112      301     478     307             98
insult          7344          1371     6155     307    7877           1160
identity_hate   1302           313     1032      98    1160           1405


## Text Cleaning Helpers

In [15]:
def clean_text(text) -> str:
    if pd.isna(text):
        return " "
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else " "


def add_clean_text(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["comment_text_clean"] = out["comment_text"].map(clean_text)
    out["comment_word_count"] = out["comment_text_clean"].str.split().str.len().fillna(0).astype(int)
    return out


def top_tokens_from_subset(subset: pd.DataFrame, n_tokens: int = 15) -> pd.Series:
    vectorizer = CountVectorizer(stop_words="english", max_features=n_tokens, token_pattern=r"(?u)\b[a-z][a-z']+\b")
    matrix = vectorizer.fit_transform(subset["comment_text_clean"])
    token_counts = matrix.sum(axis=0).A1
    return pd.Series(token_counts, index=vectorizer.get_feature_names_out()).sort_values(ascending=True)

## Train EDA

In [11]:
train_working = add_clean_text(train_labeled)
train_working = train_working.loc[:, [*train_raw.columns, "toxic_label", "comment_text_clean", "comment_word_count"]]

print("Clean text checks for the training set:")
print(f"null comment_text_clean: {train_working['comment_text_clean'].isna().sum()}")
print(f"empty comment_text_clean: {(train_working['comment_text_clean'] == '').sum()}")
print(f"binary toxic_label values: {sorted(train_working['toxic_label'].unique().tolist())}")
print()

plot_df = train_working.assign(class_name=train_working['toxic_label'].map({0: 'clean', 1: 'toxic'}))
class_order = ['clean', 'toxic']
class_counts = plot_df['class_name'].value_counts().reindex(class_order)

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax, palette=['#3b82f6', '#ef4444'])
ax.set_title('Class Balance in the Training Set')
ax.set_xlabel('Class')
ax.set_ylabel('Comment count')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'class_balance_bar_chart.png', dpi=300, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=plot_df, x='class_name', y='comment_word_count', order=class_order, ax=ax, inner='quartile', palette=['#3b82f6', '#ef4444'], cut=0)
ax.set_title('Comment Length by Class')
ax.set_xlabel('Class')
ax.set_ylabel('Word count after cleaning')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'comment_length_by_class_distribution.png', dpi=300, bbox_inches='tight')
plt.close(fig)

for class_id, class_name, color in [(0, 'clean', '#3b82f6'), (1, 'toxic', '#ef4444')]:
    subset = train_working.loc[train_working['toxic_label'] == class_id, ['comment_text_clean']]
    token_counts = top_tokens_from_subset(subset, n_tokens=15)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(token_counts.index, token_counts.values, color=color, alpha=0.9)
    ax.set_title(f'Top Tokens for the {class_name} Class')
    ax.set_xlabel('Token count')
    ax.set_ylabel('Token')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'top_tokens_{class_name}_bar_chart.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(co_occurrence, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Six-Label Co-occurrence Heatmap')
ax.set_xlabel('Label')
ax.set_ylabel('Label')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'six_label_cooccurrence_heatmap.png', dpi=300, bbox_inches='tight')
plt.close(fig)

print('Saved figures:')
for figure_path in sorted(FIGURES_DIR.glob('*.png')):
    print(f'  {figure_path.name}')

Clean text checks for the training set:
null comment_text_clean: 0
empty comment_text_clean: 0
binary toxic_label values: [0, 1]



/tmp/ipykernel_5225/163558677.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax, palette=['#3b82f6', '#ef4444'])
/tmp/ipykernel_5225/163558677.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_df, x='class_name', y='comment_word_count', order=class_order, ax=ax, inner='quartile', palette=['#3b82f6', '#ef4444'], cut=0)


Saved figures:
  class_balance_bar_chart.png
  combined_roc_curve.png
  comment_length_by_class_distribution.png
  lr_confusion_matrix.png
  minilm_lr_confusion_matrix.png
  six_label_cooccurrence_heatmap.png
  summary_metric_comparison.png
  top_tokens_clean_bar_chart.png
  top_tokens_toxic_bar_chart.png
  xgb_confusion_matrix.png


## Labeled Test Preparation

In [12]:
test_labels_labeled = test_labels_raw.loc[test_labels_raw[LABEL_COLS].ne(-1).all(axis=1)].copy()
test_prepared = test_raw.merge(test_labels_labeled[["id", *LABEL_COLS]], on='id', how='inner', validate='one_to_one')
test_prepared = add_binary_target(test_prepared)
test_prepared = add_clean_text(test_prepared)

print(f"Filtered labeled test rows: {test_labels_labeled.shape[0]} of {test_labels_raw.shape[0]}")
print(f"Merged held-out test set shape: {test_prepared.shape}")
print(pd.Series(test_prepared['toxic_label'].value_counts(normalize=True).sort_index()).rename(index={0: 'clean', 1: 'toxic'}).rename('rate'))
print(f"null comment_text_clean: {test_prepared['comment_text_clean'].isna().sum()}")
print(f"binary toxic_label values: {sorted(test_prepared['toxic_label'].unique().tolist())}")

Filtered labeled test rows: 63978 of 153164
Merged held-out test set shape: (63978, 11)
toxic_label
clean    0.90242
toxic    0.09758
Name: rate, dtype: float64
null comment_text_clean: 0
binary toxic_label values: [0, 1]


## Stratified Train / Validation Split

In [13]:
train_base = train_working[PROCESSED_COLUMNS].copy()
train_set, val_set = train_test_split(
    train_base,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_base['toxic_label'],
)

train_set = train_set.reset_index(drop=True)
val_set = val_set.reset_index(drop=True)
test_set = test_prepared[PROCESSED_COLUMNS].copy().reset_index(drop=True)

print(f"train_set shape: {train_set.shape}")
print(f"val_set shape: {val_set.shape}")
print(f"test_set shape: {test_set.shape}")
print()

summary = pd.DataFrame({
    'full_train': train_base['toxic_label'].value_counts(normalize=True).sort_index(),
    'train_set': train_set['toxic_label'].value_counts(normalize=True).sort_index(),
    'val_set': val_set['toxic_label'].value_counts(normalize=True).sort_index(),
}).rename(index={0: 'clean', 1: 'toxic'})
print('Class-rate comparison:')
print(summary)

assert train_set.shape[0] + val_set.shape[0] == train_base.shape[0]
assert set(train_set.columns) == set(PROCESSED_COLUMNS)
assert set(val_set.columns) == set(PROCESSED_COLUMNS)
assert set(test_set.columns) == set(PROCESSED_COLUMNS)

train_set shape: (127656, 3)
val_set shape: (31915, 3)
test_set shape: (63978, 3)

Class-rate comparison:
             full_train  train_set   val_set
toxic_label                                 
clean          0.898321    0.89832  0.898324
toxic          0.101679    0.10168  0.101676


## Export Checks and CSV Saves

In [14]:
def check_processed_frame(name: str, df: pd.DataFrame, expected_rows: int | None = None) -> None:
    print(f"{name}: shape={df.shape}")
    print(f"  columns={list(df.columns)}")
    print(f"  null comment_text_clean={df['comment_text_clean'].isna().sum()}")
    print(f"  toxic_label values={sorted(df['toxic_label'].unique().tolist())}")
    print(f"  toxic rate={df['toxic_label'].mean():.4f}")
    if expected_rows is not None:
        assert df.shape[0] == expected_rows, f"{name} expected {expected_rows} rows, got {df.shape[0]}"
    assert df.columns.tolist() == PROCESSED_COLUMNS, f"{name} has unexpected columns"
    assert df['comment_text_clean'].notna().all(), f"{name} has null cleaned text"
    assert set(df['toxic_label'].unique()).issubset({0, 1}), f"{name} has non-binary labels"

check_processed_frame('train_set (in memory)', train_set, expected_rows=train_set.shape[0])
check_processed_frame('val_set (in memory)', val_set)
check_processed_frame('test_set (in memory)', test_set)

expected_counts = {
    'train_set.csv': train_set,
    'val_set.csv': val_set,
    'test_set.csv': test_set,
}

for filename, frame in expected_counts.items():
    output_path = PROCESSED_DIR / filename
    frame.to_csv(output_path, index=False)
    reloaded = pd.read_csv(output_path)
    print(f"Saved {filename} -> {output_path}")
    print(f"  shape={reloaded.shape}")
    print(f"  toxic rate={reloaded['toxic_label'].mean():.4f}")
    print(f"  null comment_text_clean={reloaded['comment_text_clean'].isna().sum()}")
    print(f"  columns={list(reloaded.columns)}")
    print()

print('Figure files present:')
for figure_path in sorted(FIGURES_DIR.glob('*.png')):
    print(f'  {figure_path.name}')

train_set (in memory): shape=(127656, 3)
  columns=['id', 'comment_text_clean', 'toxic_label']
  null comment_text_clean=0
  toxic_label values=[0, 1]
  toxic rate=0.1017
val_set (in memory): shape=(31915, 3)
  columns=['id', 'comment_text_clean', 'toxic_label']
  null comment_text_clean=0
  toxic_label values=[0, 1]
  toxic rate=0.1017
test_set (in memory): shape=(63978, 3)
  columns=['id', 'comment_text_clean', 'toxic_label']
  null comment_text_clean=0
  toxic_label values=[0, 1]
  toxic rate=0.0976
Saved train_set.csv -> /content/drive/MyDrive/BT5151_toxic_comment_agent/experiments/processed_data/train_set.csv
  shape=(127656, 3)
  toxic rate=0.1017
  null comment_text_clean=0
  columns=['id', 'comment_text_clean', 'toxic_label']

Saved val_set.csv -> /content/drive/MyDrive/BT5151_toxic_comment_agent/experiments/processed_data/val_set.csv
  shape=(31915, 3)
  toxic rate=0.1017
  null comment_text_clean=0
  columns=['id', 'comment_text_clean', 'toxic_label']

Saved test_set.csv -> /